# MEDISCOPE — 01 Data Exploration

## Exploratory analysis of the HIV treatment-programme dataset

This notebook provides the first analytical view of the dataset used to develop the MEDISCOPE Loss to Follow-Up (LTFU) prediction models.

### Objectives

The notebook is designed to:

- confirm the source dataset can be loaded reproducibly;
- document its dimensions and schema;
- assess completeness and data types;
- explore key demographic, geographical, treatment and clinical variables;
- identify obvious quality issues before transformation;
- avoid exposing unnecessary patient identifiers during exploratory reporting;
- record modelling implications that should be addressed during preprocessing and feature engineering.

> **Data-governance note:** the research dataset used for model development is distinct from the synthetic patient records used by the MEDISCOPE web application. Raw patient-level research data should not be published with the repository unless data-sharing permissions explicitly allow it.

## 1. Reproducible project setup

The notebook locates the repository root dynamically, so it can be launched from either the project root or the `notebooks/` directory without hard-coded local Windows paths.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def find_project_root(start: Path | None = None) -> Path:
    """Locate the MEDISCOPE repository root from common notebook launch locations."""
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "api").is_dir()
            and (candidate / "requirements.txt").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Unable to locate the MEDISCOPE repository root. "
        "Run this notebook from the repository or notebooks directory."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "trained"
REPORT_DIR = PROJECT_ROOT / "reports" / "evaluation"

print(f"Project root: {PROJECT_ROOT}")

## 2. Load the raw source dataset

The original source file is preserved under `data/raw/`. Exploration is read-only: no transformation in this notebook overwrites the source workbook.

In [ ]:
RAW_FILE = RAW_DIR / "LTFU in HIV DataSet NDR.xlsx"

if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at {RAW_FILE}. "
        "The research dataset is intentionally not assumed to be downloadable from GitHub."
    )

df = pd.read_excel(RAW_FILE)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

The validated project dataset contains **304,273 records**. The value printed above should match that baseline unless the underlying research extract has changed.

## 3. Schema overview

A compact schema table is preferable to printing every row. It captures each column's type, number of populated values, missingness and cardinality.

In [ ]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
    "unique": df.nunique(dropna=True),
}).sort_values(["missing_pct", "unique"], ascending=[False, False])

schema

## 4. Privacy-conscious data preview

The exploratory preview intentionally prioritises non-identifying analytical fields instead of printing all source columns.

In [ ]:
candidate_preview_columns = [
    "State",
    "LGA",
    "Sex",
    "Date Of Birth",
    "Age at ART Initiation",
    "Current Age",
    "ART Start Date",
    "Days Of ARV Refill",
    "Last Regimen",
    "Pregnancy Status",
    "Current Viral Load",
    "Patient Transferred In",
    "Current Status (28 Days)",
    "Current Status (90 Days)",
]

preview_columns = [
    column for column in candidate_preview_columns
    if column in df.columns
]

df[preview_columns].head(10)

## 5. Missingness profile

Missingness is clinically and analytically important. In longitudinal programme data, an absent measurement can represent several different processes: a genuinely missing record, a measurement that was not due, incomplete reporting, transfer history, or loss of contact.

The notebook therefore quantifies missingness rather than silently dropping incomplete records.

In [ ]:
missingness = (
    df.isna()
      .mean()
      .mul(100)
      .rename("missing_pct")
      .to_frame()
      .assign(missing_count=df.isna().sum())
      .sort_values("missing_pct", ascending=False)
)

missingness.head(30)

In [ ]:
plot_missing = missingness.head(20).sort_values("missing_pct")

plt.figure(figsize=(10, 7))
plt.barh(plot_missing.index, plot_missing["missing_pct"])
plt.xlabel("Missing values (%)")
plt.ylabel("Field")
plt.title("Twenty source fields with the highest missingness")
plt.tight_layout()
plt.show()

### Interpretation

The preprocessing work later confirmed substantial incompleteness in several longitudinal viral-load and historical date fields. This supports the decision to retain explicit missingness information where clinically meaningful rather than treating every absence as random noise.

## 6. Geography

Geographical variables are useful both for understanding the population represented in the source data and for recognising that model performance may be dataset/geography specific.

In [ ]:
for column in ["State", "LGA"]:
    if column in df.columns:
        counts = df[column].fillna("Missing").value_counts().head(20)
        display(counts.to_frame("records"))

        plt.figure(figsize=(10, 5))
        counts.sort_values().plot(kind="barh")
        plt.title(f"Most frequent values — {column}")
        plt.xlabel("Records")
        plt.ylabel(column)
        plt.tight_layout()
        plt.show()

## 7. Demographic variables

In [ ]:
if "Sex" in df.columns:
    display(
        df["Sex"]
        .fillna("Missing")
        .value_counts(dropna=False)
        .to_frame("records")
    )

age_columns = [
    column for column in ["Current Age", "Age at ART Initiation"]
    if column in df.columns
]

for column in age_columns:
    values = pd.to_numeric(df[column], errors="coerce")
    print(f"\n{column}")
    display(values.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame().T)

    plt.figure(figsize=(9, 4))
    plt.hist(values.dropna(), bins=50)
    plt.xlabel("Age (years)")
    plt.ylabel("Records")
    plt.title(f"Distribution of {column}")
    plt.tight_layout()
    plt.show()

The raw `Age at ART Initiation` variable is examined again during preprocessing because source ages can contain implausible negative or extreme values.

## 8. Treatment and refill characteristics

In [ ]:
if "Days Of ARV Refill" in df.columns:
    refill = pd.to_numeric(df["Days Of ARV Refill"], errors="coerce")

    display(
        refill.describe(
            percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
        ).to_frame("Days Of ARV Refill")
    )

    plt.figure(figsize=(9, 4))
    plt.hist(refill.dropna().clip(upper=refill.quantile(0.99)), bins=50)
    plt.xlabel("Days supplied")
    plt.ylabel("Records")
    plt.title("ARV refill duration (values clipped at the 99th percentile for display)")
    plt.tight_layout()
    plt.show()

if "Last Regimen" in df.columns:
    display(
        df["Last Regimen"]
        .fillna("Missing")
        .value_counts()
        .head(20)
        .to_frame("records")
    )

## 9. Viral-load characteristics

In [ ]:
if "Current Viral Load" in df.columns:
    viral_load = pd.to_numeric(df["Current Viral Load"], errors="coerce")

    display(
        viral_load.describe(
            percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
        ).to_frame("Current Viral Load")
    )

    print(f"Missing current viral load: {viral_load.isna().sum():,}")
    print(f"Missing current viral load (%): {viral_load.isna().mean() * 100:.2f}%")

## 10. Treatment-status fields

Status fields require particular caution because some status variables may contribute to target construction. Exploratory inspection is useful, but target-related information must not be allowed to leak into the predictor matrix.

In [ ]:
status_columns = [
    column for column in df.columns
    if "Status" in str(column)
]

print(f"Status-like fields found: {len(status_columns)}")

for column in status_columns[:12]:
    print(f"\n{column}")
    display(
        df[column]
        .fillna("Missing")
        .value_counts(dropna=False)
        .head(15)
        .to_frame("records")
    )

## 11. Exploratory findings and modelling implications

The EDA stage establishes several important requirements for the later pipeline:

1. **Dates must be standardised** before temporal features such as treatment duration can be trusted.
2. **Age values require plausibility checks**, particularly `Age at ART Initiation`.
3. **Missingness is not negligible**, especially for some viral-load and longitudinal date fields.
4. **Geography and regimen are high-cardinality categorical domains** and require controlled encoding.
5. **Target/status variables require explicit leakage protection**.
6. **Clinical predictors should be transformed reproducibly**, with the final feature order persisted for inference.
7. **The raw research dataset should remain immutable**; processed artefacts belong under `data/processed/`.

### Next notebook

`02_preprocessing.ipynb` performs the reproducible cleaning and date-validation stage before feature engineering.